
# ADAM vs L-BFGS

Main notebook for comparing performance

**Note:** If you encounter broadcasting errors, restart the kernel to clear JAX's JIT cache.

In [4]:
import numpy as np
import jax
import jax.numpy as jnp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time

from kappaeta import AdamOptimizer, LBFGSOptimizer, loss_function_jax
from datasets import DATASETS

# Set random seeds
np.random.seed(42) 
key = jax.random.PRNGKey(42)

# Style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

In [2]:
# Datasets and init strategies are imported from datasets.py
print(f"Available datasets: {len(DATASETS)}")
for name in DATASETS:
    print(f"  - {name}")

Available datasets: 10
  - Friedman2
  - Friedman1
  - Moons
  - Regression
  - Polynomial Regression
  - UCI: Energy Efficiency
  - UCI: Wine
  - UCI: Heart Failure
  - Forest Fires
  - Concrete


## Benchmark

In [3]:
def run_benchmark(dataset_name, X, y, kappa_init, eta_init, optimizer_name, optimizer):
    """
    Run a single benchmark experiment.
    
    Returns:
        dict: Results including MSE, runtime, optimal parameters, and convergence info
    """
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Scale features
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)
    y_train = y_train.astype(np.float32)
    y_test = y_test.astype(np.float32)
    
    # Run optimization
    start_time = time.time()
    try:
        kappa_opt, eta_opt, history = optimizer.optimize(
            kappa_init, eta_init, X_test, X_train, y_train, y_test
        )
        runtime = time.time() - start_time
        
        # Calculate final MSE
        final_loss = history[-1]['loss']
        converged = len(history) < optimizer.max_iters
        
        # Convert kappa to array for consistent handling
        kappa_opt_array = np.array(kappa_opt)
        
        return {
            'dataset': dataset_name,
            'optimizer': optimizer_name,
            'kappa_init': f"{np.mean(kappa_init):.2f}±{np.std(kappa_init):.2f}",
            'eta_init': f"{eta_init:.2f}",
            'kappa_optimal': kappa_opt_array,
            'kappa_opt_mean': np.mean(kappa_opt_array),
            'kappa_opt_std': np.std(kappa_opt_array),
            'eta_optimal': float(eta_opt),
            'final_mse': final_loss,
            'runtime_sec': runtime,
            'iterations': len(history),
            'converged': converged,
            'time_per_iter_ms': runtime / len(history) * 1000,
            'n_features': X.shape[1]
        }
    except Exception as e:
        print(f"Error in {optimizer_name} on {dataset_name}: {str(e)}")
        return None


# Define wide range of initialization strategies
INIT_STRATEGIES = {
    '01': (1.0, 1.0),
    '02': (5.0, 5.0),
    '03': (10.0, 10.0),
    '04': (20.0, 20.0),
    '05': (30.0, 30.0),
    '06': (40.0, 40.0),
    '07': (50.0, 50.0),
    '08': (60.0, 60.0),
    '09': (70.0,70.0)


}

print("Benchmark configuration:")
print(f"  Datasets: {len(DATASETS)}")
print(f"  Initialization strategies: {len(INIT_STRATEGIES)}")
print(f"  Optimizers: 2 (Adam, L-BFGS)")
print(f"  Total experiments: {len(DATASETS) * len(INIT_STRATEGIES) * 2}")

Benchmark configuration:
  Datasets: 10
  Initialization strategies: 9
  Optimizers: 2 (Adam, L-BFGS)
  Total experiments: 180


In [ ]:
# Run comprehensive benchmark
results = []
tol = 0.00001
KAPPA_BOUND, ETA_BOUND = (1, 80.0),(1, 80.0)

for dataset_name, dataset_fn in DATASETS.items():
    print(f"\n{'='*70}")
    print(f"Dataset: {dataset_name}")
    print(f"{'='*70}")
    
    # Generate dataset
    X, y = dataset_fn()
    n_features = X.shape[1]
    
    for init_name, (kappa_val, eta_val) in INIT_STRATEGIES.items():
        # Handle random initialization
        if kappa_val is None:
            np.random.seed(42)
            kappa_init = np.random.uniform(0.5, 5.0, size=n_features)
            eta_init = np.random.uniform(0.5, 5.0)
        else:
            kappa_init = np.full(n_features, kappa_val)
            eta_init = eta_val
        
        print(f"  Init: {init_name} (κ_mean={np.mean(kappa_init):.2f}, η={eta_init:.2f})")
        
        # Adam optimizer
        adam = AdamOptimizer(
            learning_rate=1.0,
            max_iters=500,
            tol=tol,
            verbose=False,
            kappa_bounds=KAPPA_BOUND, 
            eta_bounds=ETA_BOUND
        )
        result_adam = run_benchmark(dataset_name, X, y, kappa_init, eta_init, 'Adam', adam)
        if result_adam:
            result_adam['init_strategy'] = init_name
            results.append(result_adam)
            print(f"    Adam:      MSE={result_adam['final_mse']:.6f}, Time={result_adam['runtime_sec']:.3f}s, Iters={result_adam['iterations']}")
        
        # L-BFGS optimizer
        lbfgs = LBFGSOptimizer(
            max_iters=200,
            tol=tol,
            memory_size=3,
            verbose=False,
            kappa_bounds=KAPPA_BOUND,
            eta_bounds=ETA_BOUND
        )
        result_lbfgs = run_benchmark(dataset_name, X, y, kappa_init, eta_init, 'L-BFGS', lbfgs)
        if result_lbfgs:
            result_lbfgs['init_strategy'] = init_name
            results.append(result_lbfgs)
            print(f"    L-BFGS:    MSE={result_lbfgs['final_mse']:.6f}, Time={result_lbfgs['runtime_sec']:.3f}s, Iters={result_lbfgs['iterations']}")

print(f"\n{'='*70}")
print(f"Benchmark complete! Total successful runs: {len(results)}")
print(f"{'='*70}")

In [ ]:
# Create comprehensive results dataframe
df_results = pd.DataFrame(results)

# Create summary table (excluding the full kappa arrays)
summary_cols = ['dataset', 'optimizer', 'init_strategy', 'final_mse', 'runtime_sec', 
                'iterations', 'time_per_iter_ms', 'kappa_opt_mean', 'kappa_opt_std', 
                'eta_optimal', 'converged', 'n_features']

df_summary = df_results[summary_cols].copy()
df_summary = df_summary.round({
    'final_mse': 5,
    'runtime_sec': 1,
    'time_per_iter_ms': 1,
    'kappa_opt_mean': 1,
    'kappa_opt_std': 1,
    'eta_optimal': 1
})

# Sort by dataset and optimizer
df_summary = df_summary.sort_values(['dataset', 'init_strategy', 'optimizer'])

df_output = df_summary[['dataset','optimizer','final_mse','runtime_sec','iterations','time_per_iter_ms','converged','n_features','init_strategy']]
print("\n" + "="*100)
print("COMPREHENSIVE BENCHMARK RESULTS")
print("="*100)
print(df_output.to_string(index=False))
print("="*100)

In [7]:
# Best MSE per dataset across all init strategies and optimizers + total search time
print("="*130)
print("BEST MSE PER DATASET (across all initializations)")
print("="*130)
print(f"{'Dataset':<35} {'Best MSE':>12} {'Optimizer':>10} {'Init (κ,η)':>14} {'Iters':>6} {'Search Time (s)':>16}")
print("-"*130)

total_search_time = sum(r['runtime_sec'] for r in results)

for ds_name in DATASETS.keys():
    ds_results = [r for r in results if r['dataset'] == ds_name]
    if not ds_results:
        continue
    
    # Total time spent searching this dataset (all inits, all optimizers)
    ds_search_time = sum(r['runtime_sec'] for r in ds_results)
    
    # Best result across all inits and optimizers
    best = min(ds_results, key=lambda x: x['final_mse'])
    
    print(f"{ds_name:<35} {best['final_mse']:>12.6f} {best['optimizer']:>10} "
          f"{best['kappa_init']:>14} {best['iterations']:>6} {ds_search_time:>16.2f}")

print("-"*130)
print(f"{'TOTAL SEARCH TIME':>35} {'':>12} {'':>10} {'':>14} {'':>6} {total_search_time:>16.2f}")
print("="*130)

BEST MSE PER DATASET (across all initializations)
Dataset                                 Best MSE  Optimizer     Init (κ,η)  Iters  Search Time (s)
----------------------------------------------------------------------------------------------------------------------------------
Friedman2                             665.386536     L-BFGS      5.00±0.00     25            68.33
Friedman1                             106.149452     L-BFGS     60.00±0.00     60            56.80
Moons                                   0.001491     L-BFGS      5.00±0.00     14            13.46
Regression                              0.604098       Adam     70.00±0.00     43             5.50
Polynomial Regression                   0.030223     L-BFGS     40.00±0.00     21             8.94
UCI: Energy Efficiency                  0.229909       Adam      5.00±0.00     41            16.04
UCI: Wine                               0.000007     L-BFGS      5.00±0.00     18             5.85
UCI: Heart Failure         

In [8]:
# ADAM vs L-BFGS comparison + best overall per dataset
print("="*130)
print("ADAM vs L-BFGS COMPARISON")
print("="*130)
print(f"{'Dataset':<35} {'Best Adam MSE':>14} {'Best L-BFGS MSE':>16} {'Winner':>10} {'Adam Time (s)':>14} {'L-BFGS Time (s)':>16}")
print("-"*130)

win_counts = {'Adam': 0, 'L-BFGS': 0}

for ds_name in DATASETS.keys():
    adam_results = [r for r in results if r['dataset'] == ds_name and r['optimizer'] == 'Adam']
    lbfgs_results = [r for r in results if r['dataset'] == ds_name and r['optimizer'] == 'L-BFGS']
    
    best_adam = min(adam_results, key=lambda x: x['final_mse']) if adam_results else None
    best_lbfgs = min(lbfgs_results, key=lambda x: x['final_mse']) if lbfgs_results else None
    
    adam_mse = best_adam['final_mse'] if best_adam else float('inf')
    lbfgs_mse = best_lbfgs['final_mse'] if best_lbfgs else float('inf')
    
    adam_total_time = sum(r['runtime_sec'] for r in adam_results)
    lbfgs_total_time = sum(r['runtime_sec'] for r in lbfgs_results)
    
    winner = 'Adam' if adam_mse <= lbfgs_mse else 'L-BFGS'
    win_counts[winner] += 1
    
    adam_str = f"{adam_mse:>14.6f}" if best_adam else f"{'N/A':>14}"
    lbfgs_str = f"{lbfgs_mse:>16.6f}" if best_lbfgs else f"{'N/A':>16}"
    
    print(f"{ds_name:<35} {adam_str} {lbfgs_str} {winner:>10} {adam_total_time:>14.2f} {lbfgs_total_time:>16.2f}")

print("-"*130)
for opt, count in win_counts.items():
    print(f"  {opt} wins: {count}/{len(DATASETS)}")

# Overall best per dataset
print("\n" + "="*130)
print("OVERALL BEST RESULT PER DATASET")
print("="*130)
print(f"{'Dataset':<35} {'Best MSE':>12} {'Optimizer':>10} {'Init':>6} {'κ_opt (mean)':>13} {'η_opt':>8} {'Time (s)':>10}")
print("-"*130)

for ds_name in DATASETS.keys():
    ds_results = [r for r in results if r['dataset'] == ds_name]
    if not ds_results:
        continue
    best = min(ds_results, key=lambda x: x['final_mse'])
    print(f"{ds_name:<35} {best['final_mse']:>12.6f} {best['optimizer']:>10} {best['init_strategy']:>6} "
          f"{best['kappa_opt_mean']:>13.4f} {best['eta_optimal']:>8.4f} {best['runtime_sec']:>10.2f}")
print("="*130)

ADAM vs L-BFGS COMPARISON
Dataset                              Best Adam MSE  Best L-BFGS MSE     Winner  Adam Time (s)  L-BFGS Time (s)
----------------------------------------------------------------------------------------------------------------------------------
Friedman2                               665.389526       665.386536     L-BFGS          51.32            17.01
Friedman1                               106.149605       106.149452     L-BFGS          44.41            12.39
Moons                                     0.002189         0.001491     L-BFGS          10.81             2.65
Regression                                0.604098         0.604131       Adam           4.30             1.20
Polynomial Regression                     0.030224         0.030223     L-BFGS           8.40             0.55
UCI: Energy Efficiency                    0.229909         0.230091       Adam          12.29             3.75
UCI: Wine                                 0.000226         0.00000

## Optimized κ and η per Dataset

In [9]:
# Table: best optimized kappa values (full list) and eta for each dataset
print("="*110)
print("OPTIMIZED κ AND η PER DATASET  (best MSE across all init strategies and optimizers)")
print("="*110)
print(f"{'Dataset':<35} {'Optimizer':>10} {'Init':>6} {'Final MSE':>12} {'η_opt':>8}  {'κ_opt (per feature)'}")
print("-"*110)

for ds_name in DATASETS.keys():
    ds_results = [r for r in results if r['dataset'] == ds_name]
    if not ds_results:
        continue
    best = min(ds_results, key=lambda x: x['final_mse'])
    kappa_list = [f"{v:.1f}" for v in best['kappa_optimal']]
    kappa_str = "[" + ", ".join(kappa_list) + "]"
    print(
        f"{ds_name:<35} {best['optimizer']:>10} {best['init_strategy']:>6} "
        f"{best['final_mse']:>12.3f} {best['eta_optimal']:>8.1f}  {kappa_str}"
    )

print("="*110)

OPTIMIZED κ AND η PER DATASET  (best MSE across all init strategies and optimizers)
Dataset                              Optimizer   Init    Final MSE    η_opt  κ_opt (per feature)
--------------------------------------------------------------------------------------------------------------
Friedman2                               L-BFGS     02      665.387      3.9  [1.0, 7.4, 5.8, 1.0]
Friedman1                               L-BFGS     08      106.149      9.5  [70.3, 46.6, 23.3, 19.3, 12.4]
Moons                                   L-BFGS     02        0.001     80.0  [3.6, 9.6]
Regression                                Adam     09        0.604     25.6  [80.0]
Polynomial Regression                   L-BFGS     06        0.030     11.5  [80.0]
UCI: Energy Efficiency                    Adam     02        0.230     12.1  [1.0, 1.0, 12.0, 6.1, 10.9, 1.0, 11.7, 1.5]
UCI: Wine                               L-BFGS     02        0.000     63.6  [15.7, 10.2, 4.7, 22.9, 20.3, 1.2, 18.5, 3.5, 1.

In [10]:
df_results.to_csv('results.csv')